# 02 — dist_autoへのGPR適用

元コードと同じく、1つのtagを丸ごとテストにして、残りtagから予測します。既定は `TEST_TAG='10'` です。予測平均に加え、不確実性と95%区間を作ります。

In [ ]:
from pathlib import Path
import os, subprocess, sys

REPO_URL = 'https://github.com/futoshi-futami/Chemistory.git'
cwd = Path.cwd()
if (cwd / 'pyproject.toml').exists():
    PROJECT_ROOT = cwd
elif (Path('/content/Chemistory') / 'pyproject.toml').exists():
    PROJECT_ROOT = Path('/content/Chemistory')
elif 'google.colab' in sys.modules:
    PROJECT_ROOT = Path('/content/Chemistory')
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(PROJECT_ROOT)], check=True)
else:
    raise FileNotFoundError('Chemistory project rootでノートブックを実行してください。')
os.chdir(PROJECT_ROOT)
subprocess.run([sys.executable, 'scripts/prepare_data.py'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(PROJECT_ROOT)], check=True)
src_dir = str(PROJECT_ROOT / 'src')
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)
import importlib
importlib.invalidate_caches()
import chemistory_gpr
print('PROJECT_ROOT =', PROJECT_ROOT)
print('chemistory_gpr =', chemistory_gpr.__file__)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from IPython.display import display
from chemistory_gpr.dist_auto import (
    DistAutoGPRConfig, fit_held_out_tag, leave_one_tag_out,
    load_dist_auto_data, make_grid, predict_grid,
)

DATA_DIR = PROJECT_ROOT / 'data' / 'dist_auto'
RESULTS = PROJECT_ROOT / 'results'
RESULTS.mkdir(exist_ok=True)
TEST_TAG = '10'
GRID_SIZE = 30

## A. データとtag外挿設定

In [ ]:
data = load_dist_auto_data(DATA_DIR)
summary = pd.DataFrame({'tag': data.tags, 'y': data.y}).groupby('tag', sort=False)['y'].agg(['count','mean','std','min','max'])
print('common Xmat features =', len(data.feature_columns))
display(summary)

`tag=b` は目的変数の平均・分散が他tagと大きく異なります。したがって、tag 10だけでなく全tagのleave-one-tag-out診断も後で確認します。

## B. 指定tagを完全にhold outしてGPRを評価

In [ ]:
config = DistAutoGPRConfig(
    name='dist_auto_full_matern32', representation='full',
    include_xy=True, matern_nu=1.5, seed=123,
)
model, heldout, metrics = fit_held_out_tag(data, TEST_TAG, config)
display(pd.DataFrame([metrics]).drop(columns='kernel'))
print('optimized kernel:', metrics['kernel'])
heldout.to_csv(RESULTS / f'dist_auto_test_{TEST_TAG}_predictions.csv', index=False)

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5))
ax.errorbar(heldout['y'], heldout['pred_mean'], yerr=1.96*heldout['pred_std'], fmt='o', ms=4, alpha=.7)
lims = [min(heldout['y'].min(), heldout['lower_95'].min()), max(heldout['y'].max(), heldout['upper_95'].max())]
ax.plot(lims, lims, '--', color='black')
ax.set(xlabel='Observed y', ylabel='Predicted y', title=f'Held-out tag {TEST_TAG}: mean ± 95%')
plt.show()

## C. leave-one-tag-out診断

In [ ]:
group_oof, group_metrics = leave_one_tag_out(data, config)
display(group_metrics[['test_tag','R2','RMSE','MAE','corr2','coverage_95','width_95']])
group_oof.to_csv(RESULTS / 'dist_auto_leave_one_tag_out_predictions.csv', index=False)
group_metrics.to_csv(RESULTS / 'dist_auto_leave_one_tag_out_metrics.csv', index=False)

`corr2` は元コードとの比較用です。主指標は `R2=1-SSE/SST` です。相関が高くても平均がずれると `corr2` は高いまま `R2` が悪化するので、両者を混同しないでください。

## D. 新しいxyグリッドの予測面と不確実性

In [ ]:
grid, grid_features = make_grid(DATA_DIR, TEST_TAG, data.feature_columns, grid_size=GRID_SIZE)
surface = predict_grid(model, grid, grid_features)
surface.to_csv(RESULTS / f'dist_auto_surface_{TEST_TAG}.csv', index=False)
mean_pivot = surface.pivot(index='y', columns='x', values='pred_mean')
std_pivot = surface.pivot(index='y', columns='x', values='pred_std')
fig = go.Figure(go.Surface(
    x=mean_pivot.columns, y=mean_pivot.index, z=mean_pivot.to_numpy(),
    surfacecolor=std_pivot.to_numpy(), colorbar={'title':'predictive std'},
))
fig.update_layout(title=f'GPR mean surface — held-out tag {TEST_TAG}', scene={'xaxis_title':'x','yaxis_title':'y','zaxis_title':'prediction'})
fig.show()

In [ ]:
fig = go.Figure(go.Heatmap(x=std_pivot.columns, y=std_pivot.index, z=std_pivot.to_numpy(), colorbar={'title':'std'}))
fig.update_layout(title='Predictive uncertainty', xaxis_title='x', yaxis_title='y')
fig.show()

best_mean = surface.loc[surface['pred_mean'].idxmax(), ['x','y','pred_mean','pred_std','lower_95']]
best_lcb = surface.loc[surface['lower_confidence_bound'].idxmax(), ['x','y','pred_mean','pred_std','lower_confidence_bound']]
print('Maximum predictive mean:\n', best_mean.to_string())
print('\nMaximum 95% lower confidence bound:\n', best_lcb.to_string())

### 回転角について

このColab版は検証可能な `angle=0` を対象にします。受領ZIPの `rotate_xyz.exe` はWindows専用で、同梱の `source.xyz` と `*-altered.xyz` はNULのみでした。非ゼロ回転を厳密に追加するには、回転プログラムのCソースまたは正常な回転前後XYZが必要です。